# Star-shaped set layout with curriculum learning

Optimizes star-shaped (radially convex) boundaries around groups of circles, with circle positions also treated as optimization variables.

The core challenge is that the loss landscape has many competing objectives — enclosure, exclusion, compactness, and collision avoidance — that conflict at different stages of training. A naive schedule that enforces all terms simultaneously leads to poor local minima.

**Curriculum learning** addresses this by introducing terms gradually via weight schedules: first pulling circles toward their sets (`set_attraction` warmup), then enforcing non-overlap (`circle_collision` warmup), and finally tightening the boundaries (`exclusion`, `area`, `perimeter` late warmup) while relaxing the attraction term (cooldown).

The notebook then shows how to **automatically search** for better schedules using Optuna.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
from jax import numpy as jnp
from vizopt.radially_convex import optimize_multiple_radially_convex_sets_with_movable_circles
from vizopt.animation import SnapshotCallback

In [ ]:
rng = np.random.default_rng(0)

n_sets = 3
n_circles_per_set = 5

In [ ]:
N = n_sets * n_circles_per_set
positions = rng.uniform(-3.0, 3.0, size=(N, 2)).astype(np.float32)
radii = rng.uniform(0.3, 0.7, size=(N, 1)).astype(np.float32)
circles = np.hstack([positions, radii])  # shape (N, 3): [cx, cy, r]

sets = [
    list(range(s * n_circles_per_set, (s + 1) * n_circles_per_set))
    for s in range(n_sets)
]

In [ ]:
# ── Optimize ──────────────────────────────────────────────────────────────────
snapshot_cb = SnapshotCallback(every=200)

n_iter_max = 6000
n_iter_phase = n_iter_max / 3

def warmup_schedule(step):
    return jnp.minimum(1.0, step / n_iter_phase)

def late_warmup_schedule(step):
    return jnp.clip((step - n_iter_phase)/n_iter_phase, min=0.01, max=1.0)

def cool_down_schedule(step):
    return jnp.clip((2*n_iter_phase - step)/n_iter_phase, min=0.01, max=1.0)

term_schedules = {"exclusion": late_warmup_schedule,
                  "area": late_warmup_schedule,
                  "perimeter": late_warmup_schedule,
                  "circle_collision": warmup_schedule,
                  "set_attraction": cool_down_schedule}

results, circles_out, history = optimize_multiple_radially_convex_sets_with_movable_circles(
    circles,
    sets,
    k_angles=64,
    weight_area=1.0,
    weight_perimeter=2.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.1,
    weight_circle_collision=50.0,
    weight_bounding_box=1.,
    weight_set_attraction=10.,
    optim_kwargs={"n_iters": 5000, "learning_rate": 0.005, "callback": snapshot_cb},
    term_schedules=term_schedules
)
print(f"Captured {len(snapshot_cb.snapshots)} snapshots")

In [ ]:
import matplotlib.cm as cm
import matplotlib.animation as manimation
from matplotlib import pyplot as plt
from IPython.display import HTML

circle_radii = np.array([c[2] for c in circles])
angles_anim = np.linspace(0, 2 * np.pi, 64, endpoint=False)
cmap_anim = cm.get_cmap("tab20", len(sets))

# Fixed axis limits: use a bounding box from final result plus margin
all_x, all_y = [], []
for result in results:
    cx, cy = result["center"]
    all_x.extend((cx + result["radii"] * np.cos(result["angles"])).tolist())
    all_y.extend((cy + result["radii"] * np.sin(result["angles"])).tolist())
margin = 0.5
xlim = (min(all_x) - margin, max(all_x) + margin)
ylim = (min(all_y) - margin, max(all_y) + margin)

order = sorted(range(len(sets)), key=lambda s: len(sets[s]), reverse=True)

def render_snapshot(optim_vars):
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")

    for s in order:
        cx, cy = optim_vars["centers"][s]
        s_radii = optim_vars["radii"][s]
        bx = np.append(cx + s_radii * np.cos(angles_anim), cx + s_radii[0] * np.cos(angles_anim[0]))
        by = np.append(cy + s_radii * np.sin(angles_anim), cy + s_radii[0] * np.sin(angles_anim[0]))
        color = cmap_anim(s)
        ax.fill(bx, by, alpha=0.10, color=color)
        ax.plot(bx, by, color=color, linewidth=1.0, alpha=0.7)

    # Draw circles at their current optimized positions
    circle_positions = optim_vars["circle_positions"]  # shape (N, 2)
    for s in order:
        color = cmap_anim(s)
        for i in sets[s]:
            cx_c, cy_c = circle_positions[i]
            r = circle_radii[i]
            patch = plt.Circle((cx_c, cy_c), r, color=color, alpha=0.4, linewidth=0.5, edgecolor=color)
            ax.add_patch(patch)

    return fig

# Build frames
images = []
for i_iter, optim_vars in snapshot_cb.snapshots:
    fig = render_snapshot(optim_vars)
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).copy()
    images.append((i_iter, buf.reshape(h, w, 4)))
    plt.close(fig)

# Assemble animation
fig_anim, ax_anim = plt.subplots()
ax_anim.axis("off")
fig_anim.tight_layout(pad=0)
im = ax_anim.imshow(images[0][1])
title = ax_anim.set_title(f"Iteration {images[0][0]}")

def update(frame):
    i_iter, img = images[frame]
    im.set_data(img)
    title.set_text(f"Iteration {i_iter}")
    return [im, title]

anim = manimation.FuncAnimation(fig_anim, update, frames=len(images), interval=150, blit=True)
HTML(anim.to_jshtml())

## Automatically finding better schedules

Each term's schedule is a function `step → scalar` that ramps the effective weight up (warmup) or down (cooldown) over training. The current schedules were designed by hand.

We can search for better schedules by:
1. **Parameterizing** each schedule by a few scalars (delay fraction, ramp fraction)
2. **Evaluating** a candidate by running the inner optimization and reading off the final constraint violations
3. **Searching** with Optuna (Bayesian TPE) over the parameter space

The quality metric is the final weighted loss of the hard constraints: enclosure + exclusion + circle collisions. Lower = better layout with satisfied constraints.

In [ ]:
def make_term_schedules(params, n_iters):
    """Build term_schedules from a flat parameter dict.

    Each warmup term has:
        {term}_delay  – fraction of n_iters before ramping starts
        {term}_ramp   – fraction of n_iters over which to ramp from 0.01 → 1.0

    The cooldown term (set_attraction) has:
        attraction_peak – fraction of n_iters at which the weight is still 1.0
        attraction_ramp – fraction of n_iters over which to ramp from 1.0 → 0.01
    """
    def warmup(delay_frac, ramp_frac):
        delay = delay_frac * n_iters
        duration = max(ramp_frac * n_iters, 1.0)
        def fn(step):
            return jnp.clip((step - delay) / duration, 0.01, 1.0)
        return fn

    def cooldown(peak_frac, ramp_frac):
        peak = peak_frac * n_iters
        duration = max(ramp_frac * n_iters, 1.0)
        def fn(step):
            return jnp.clip((peak - step) / duration + 1.0, 0.01, 1.0)
        return fn

    return {
        "circle_collision": warmup(params["collision_delay"], params["collision_ramp"]),
        "exclusion":        warmup(params["exclusion_delay"], params["exclusion_ramp"]),
        "area":             warmup(params["area_delay"],      params["area_ramp"]),
        "perimeter":        warmup(params["perimeter_delay"], params["perimeter_ramp"]),
        "set_attraction":   cooldown(params["attraction_peak"], params["attraction_ramp"]),
    }


# set_attraction is a relaxation term – it should finish at 0 and is excluded from quality
_QUALITY_TERMS = {"enclosure", "exclusion", "min_radius", "smoothness",
                  "area", "perimeter", "position_anchor", "circle_collision", "bounding_box"}


def evaluate_schedules(params, n_iters=2000):
    """Run the optimization with the given schedule params and return a quality score.

    Lower is better. Score = sum of all final weighted losses except set_attraction
    (schedule=1 for all quality terms at the end; set_attraction is a relaxation
    term that is expected to cool down to 0 and should not count toward quality).
    """
    schedules = make_term_schedules(params, n_iters)
    _, _, history = optimize_multiple_radially_convex_sets_with_movable_circles(
        circles,
        sets,
        k_angles=32,                    # coarser during search for speed
        weight_area=1.0,
        weight_perimeter=2.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_position_anchor=0.1,
        weight_circle_collision=50.0,
        weight_bounding_box=1.0,
        weight_set_attraction=10.0,
        term_schedules=schedules,
        optim_kwargs={"n_iters": n_iters, "learning_rate": 0.005},
    )
    final = history[-1]
    score = sum(final[k] for k in _QUALITY_TERMS if k in final)
    return float(score), final

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 40
SEARCH_ITERS = 2000  # cheaper inner run during search

def objective(trial):
    params = {
        "collision_delay": trial.suggest_float("collision_delay", 0.0, 0.5),
        "collision_ramp":  trial.suggest_float("collision_ramp",  0.05, 0.6),
        "exclusion_delay": trial.suggest_float("exclusion_delay", 0.0, 0.7),
        "exclusion_ramp":  trial.suggest_float("exclusion_ramp",  0.05, 0.6),
        "area_delay":      trial.suggest_float("area_delay",      0.0, 0.7),
        "area_ramp":       trial.suggest_float("area_ramp",       0.05, 0.6),
        "perimeter_delay": trial.suggest_float("perimeter_delay", 0.0, 0.7),
        "perimeter_ramp":  trial.suggest_float("perimeter_ramp",  0.05, 0.6),
        "attraction_peak": trial.suggest_float("attraction_peak", 0.2, 0.9),
        "attraction_ramp": trial.suggest_float("attraction_ramp", 0.05, 0.6),
    }
    score, _ = evaluate_schedules(params, n_iters=SEARCH_ITERS)
    return score

# Seed the study with the manually designed schedules as trial 0
manual_params = {
    "collision_delay": 0.0,   "collision_ramp":  1/3,
    "exclusion_delay": 1/3,   "exclusion_ramp":  1/3,
    "area_delay":      1/3,   "area_ramp":       1/3,
    "perimeter_delay": 1/3,   "perimeter_ramp":  1/3,
    "attraction_peak": 2/3,   "attraction_ramp": 1/3,
}

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42))
study.enqueue_trial(manual_params)  # start from the known-good point
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f"Best score: {best.value:.4f}  (manual baseline: {study.trials[0].value:.4f})")
print("Best params:", best.params)

In [ ]:
# Plot the schedule curves for the manual baseline vs the best found
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
steps = np.arange(SEARCH_ITERS)

for ax, label, params in [
    (axes[0], "Manual", manual_params),
    (axes[1], f"Best (Optuna, score={best.value:.2f})", best.params),
]:
    schedules = make_term_schedules(params, SEARCH_ITERS)
    for name, fn in schedules.items():
        vals = np.array([float(fn(jnp.int32(s))) for s in steps])
        ax.plot(steps, vals, label=name)
    ax.set_title(label)
    ax.set_xlabel("step")
    ax.set_ylabel("schedule weight")
    ax.legend(fontsize=8)
    ax.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.show()

In [ ]:
# Re-run the best schedule at full resolution (k_angles=64, n_iters=5000)
best_schedules = make_term_schedules(best.params, n_iters=5000)
snapshot_cb_best = SnapshotCallback(every=200)

results_best, circles_out_best, history_best = optimize_multiple_radially_convex_sets_with_movable_circles(
    circles,
    sets,
    k_angles=64,
    weight_area=1.0,
    weight_perimeter=2.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.1,
    weight_circle_collision=50.0,
    weight_bounding_box=1.0,
    weight_set_attraction=10.0,
    term_schedules=best_schedules,
    optim_kwargs={"n_iters": 5000, "learning_rate": 0.005, "callback": snapshot_cb_best},
)

score_best, _ = evaluate_schedules(best.params, n_iters=SEARCH_ITERS)
score_manual, _ = evaluate_schedules(manual_params, n_iters=SEARCH_ITERS)
print(f"Manual schedule score:  {score_manual:.4f}")
print(f"Best schedule score:    {score_best:.4f}")
print(f"Improvement: {(score_manual - score_best) / score_manual * 100:.1f}%")

In [ ]:
cmap = plt.cm.get_cmap("tab20", len(sets))
angles_plot = np.linspace(0, 2 * np.pi, 64, endpoint=False)

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_aspect("equal")
ax.axis("off")

order = sorted(range(len(sets)), key=lambda s: len(sets[s]), reverse=True)

for s in order:
    color = cmap(s)
    res = results_best[s]
    cx, cy = res["center"]
    bx = np.append(cx + res["radii"] * np.cos(res["angles"]), cx + res["radii"][0] * np.cos(res["angles"][0]))
    by = np.append(cy + res["radii"] * np.sin(res["angles"]), cy + res["radii"][0] * np.sin(res["angles"][0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=1.5)

for s in order:
    color = cmap(s)
    for i in sets[s]:
        cx_c, cy_c, r = circles_out_best[i]
        patch = plt.Circle((cx_c, cy_c), r, color=color, alpha=0.5, linewidth=0.8, edgecolor=color)
        ax.add_patch(patch)

plt.tight_layout()
plt.show()